<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

## Inequality Constraints and Active Bounds

**Determine which calibrated clock settings remain acceptable when an upper bound is added.**

Lecture 03-2 used Lagrange multipliers for exact calibration. We retain its squared-error objective and restore the original calibration target of 1 min at one hour. We now add an upper bound on the initial correction.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 1 · Retain calibration and impose an upper bound

The observed errors are $d_i=-2,-1,-1,0$ min at $t_i=0,1,2,3$ h, for $i=1,\ldots,4$. Choose initial correction $q$ in min and rate correction $r$ in min/h. The remaining errors are $y_i=d_i+q+rt_i$ and the squared-error objective is

$$
\phi(q,r)=\sum_{i=1}^{4}y_i^2
=4q^2+12qr+14r^2-8q-6r+6.
$$

The original calibration requirement at one hour is $q+(1\,\mathrm h)r=1\,\mathrm{min}$. Using numerical values in these units, keep the equality residual $h(q,r)=q+r-1=0$. Let $q_{\max}$ be the upper limit on the initial correction, in minutes, and define the inequality residual $g(q,r)=q-q_{\max}\le0$.

Unless a comparison states otherwise, use $q_{\max}=1$. The equality-only solution $(1.5,-0.5)$ violates this upper bound. The next definition keeps physical errors, the objective, and feasibility checks separate.


In [ ]:
import numpy as np

# Fixed observations and their elapsed times
OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])  # min
OBSERVATION_TIMES = np.array([0.0, 1.0, 2.0, 3.0])   # h

# Calibration requirement: correction at 1 h must equal 1 min
CALIBRATION_TIME = 1.0        # h
REQUIRED_CORRECTION = 1.0     # min
CHECK_TOLERANCE = 1e-9        # numerical comparison setting


def simulate_clock(decision):
    correction, rate = np.asarray(decision, dtype=float)
    return OBSERVED_ERRORS + correction + rate * OBSERVATION_TIMES


def evaluate_clock(decision):
    decision = np.asarray(decision, dtype=float)
    remaining_errors = simulate_clock(decision)
    correction, rate = decision
    equality_residual = correction + CALIBRATION_TIME * rate - REQUIRED_CORRECTION
    gradient = 2 * np.array([np.sum(remaining_errors),
                             np.dot(OBSERVATION_TIMES, remaining_errors)])
    return {
        "decision": decision,
        "response": remaining_errors,
        "objective": float(np.sum(remaining_errors ** 2)),
        "gradient": gradient,
        "equality_residual": float(equality_residual),
        "feasible": bool(abs(equality_residual) <= CHECK_TOLERANCE),
    }


CORRECTION_LIMIT = 1.0  # min


def evaluate_bounded_clock(decision, correction_limit=CORRECTION_LIMIT):
    result = evaluate_clock(decision)
    inequality_residual = float(result["decision"][0] - correction_limit)
    result.update({
        "inequality_residual": inequality_residual,
        "feasible": bool(result["feasible"] and inequality_residual <= CHECK_TOLERANCE),
    })
    return result


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

Compare the equality-only answer with two feasible decisions. Reject any violated requirement before comparing scores. For example, $(q,r)=(0.8,0)$ satisfies $q\le1$ but has equality residual $0.8+0-1=-0.2$, so it is rejected. Both requirements must hold.

This finite comparison does not prove a minimum over the whole feasible set.


In [ ]:
for decision in ([1.5, -0.5], [1.0, 0.0], [0.0, 1.0]):
    result = evaluate_bounded_clock(decision)
    if not result["feasible"]:
        print(f"{decision}: reject | h = {result['equality_residual']:.2f}"
              f" | g = {result['inequality_residual']:.2f}")
        continue
    print(f"{decision}: feasible | score = {result['objective']:.2f} min²")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 2 · Identify active and inactive inequalities

An inequality is **active** when its residual is zero and **inactive** when its residual is negative. For the upper bound, this means

$$
g=0\ \Longleftrightarrow\ q=q_{\max},\qquad
g<0\ \Longleftrightarrow\ q<q_{\max}.
$$

With $q_{\max}=1$, the calibrated settings $(1,0)$ are on the bound. The settings $(0.8,0.2)$ satisfy calibration because $0.8+0.2=1$, and their inequality residual is $g=0.8-1=-0.2$. Both settings are feasible, but only $(1,0)$ makes the inequality active.

Changing an upper bound changes feasibility, not the score of a fixed setting. An equality must always hold exactly; an inequality may have room below its limit.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 3 · Distinguish a tight bound from a loose one

Hold the equality-only solution $(1.5,-0.5)$ fixed and change only the requirement:

| Upper bound $q_{\max}$ (min) | Residual $g$ at the equality-only solution | Status |
|:---|---:|:---|
| $1$ | $0.5$ | Infeasible |
| $1.5$ | $0$ | Feasible; active bound |
| $2$ | $-0.5$ | Feasible; inactive bound |

When this solution is feasible, its equality-constrained minimum score of 0.5 min² remains attainable. When it is rejected, another feasible setting must be found. KKT conditions provide equations for those candidates.


In [ ]:
for limit in (1.0, 1.5, 2.0):
    result = evaluate_bounded_clock([1.5, -0.5], limit)
    print(f"Limit {limit:.1f} min: residual {result['inequality_residual']:.1f}, "
          f"feasible = {result['feasible']}")
